## 3. Data Preparation

Two rules govern every feature in this table:

- **Leakage-aware lags.** Every macro predictor enters as a *lag*, never contemporaneously
  — a forecast made "as of" quarter $t$ only ever sees data that would genuinely have been
  published by then (`src/features.py`).
- **Documented interventions, not silent dummies.** Structural breaks (GST, COVID) are
  captured as named dummy columns with a cited reason and a genuine forecast-lead-time
  count — not fitted after the fact to explain away a residual.

**A caveat on "as of" data.** The leakage-aware lag rule above is real and enforced in
code, but it rests on a transparent assumption set, not an audited real-time record: the
curated dataset is built from revised historical ABS/RBA/market data — the values
statistical agencies publish today for past quarters — not true real-time release
vintages, the values that would genuinely have been available on each historical
publication date. `data/metadata/series_availability.csv` documents the lag assumption
used for each series, but a genuine real-time vintage archive would be needed to fully
audit it.

*Source: [Appendix §9 — Leakage And Forecast-Origin Availability Assumptions](appendix_eda/09_leakage_availability)*

**Related appendix sections.** Which macro features actually survive into Elastic Net, and why regularizing them matters, is covered in detail in the appendix:

- [§8 Multicollinearity And VIF](appendix_eda/08_multicollinearity_vif): only `wpi_growth_lag2` crosses the VIF=10 threshold (12.7), with `inflation_expectations_business_lag1` close behind (9.6) — real but moderate redundancy among the external macro features, which is the concrete justification for Elastic Net's $\ell_2$ penalty.
- [§10 Candidate Exogenous Feature Screening — headline](appendix_eda/10_exogenous_screening_headline): nine features survive screening, led by `inflation_expectations_business_lag1`, `ppi_growth_lag2`, and the cash-rate lags — a shortlist, not the final feature set, which is still chosen by walk-forward RMSE in §4.2.
- [§11 Candidate Exogenous Feature Screening — trimmed mean](appendix_eda/11_exogenous_screening_trimmed_mean): re-running the screen on `trimmed_mean_cpi_yoy` surfaces a materially different mix — inflation-expectations and cash-rate correlations weaken sharply while commodity/oil-price features move to the front — the reason headline and trimmed-mean Elastic Net use different feature sets.

In [39]:
interventions = pd.read_csv(PROJECT_ROOT / "data/metadata/intervention_quarters.csv")
interventions[["quarter", "dummy_name", "lead_quarters", "reason"]].assign(
    reason=lambda d: d["reason"].str.slice(0, 90) + "…"
)

,quarter,dummy_name,lead_quarters,reason
0,2020Q2,covid_shock_down,0,"CPI fell 1.9% q/q, the largest fall on record,..."
1,2020Q3,covid_shock_rebound,1,"CPI rose 1.6% q/q, the largest rise in 20 year..."


**Interpretation.** Only two structural breaks get a named dummy, both from the 2020
COVID shock: `covid_shock_down` (2020Q2, CPI's largest quarterly fall on record) and
`covid_shock_rebound` (2020Q3, its largest rise in 20 years). The `lead_quarters` column
shows the leakage-aware rule applied even to the dummies themselves — `covid_shock_down` is
available with zero lead (the fall was already visible within the reference quarter), while
`covid_shock_rebound` carries a one-quarter lead, since by the time it's used as a predictor
for a *future* forecast, the prior quarter's rebound is already known. Neither the GST
introduction (1999-2000) nor the 2022-23 inflation surge gets a dummy — those episodes are
left for the models (and the [§3 CPI Target Inspection z-score screen](appendix_eda/03_cpi_target_inspection))
to explain through their own dynamics rather than being assumed away.


**A note on where else "the COVID shock" appears.** These two dummies are defined on the
raw quarter-on-quarter CPI print. The [§4.4 SVAR](04_modeling/44_svar) shock-location
export (`reports/tableau/svar_shock_events.csv`) does *not* flag `2020Q2`/`2020Q3` — it
z-scores the structural shock to the **year-on-year** `cpi_yoy`/`trimmed_mean_cpi_yoy`
series instead, fit full-sample with no COVID dummy or exclusion. A YoY series smooths the
2020 level shock across four quarters, so the same event only reads as statistically
unusual a year later, once the depressed 2020 base drops out of the comparison — which is
why that export instead flags `2021Q2`/`2021Q3`. Same event, two different transforms of
CPI, not a discrepancy between the two artifacts.